# Lab 0702 · Inmutable, perezoso y el plan
### Tema 07 · Spark I · Fundamentos

Antes de empezar, ejecuta **`00_preparar_entorno`** (crea el catálogo `lab0702`).

Trabajamos en **serverless**: `spark` ya existe, no lo crees. Para ver qué ha hecho un job: **See performance → Query Profile**.

**El hilo:** contar las palabras de *Orgullo y prejuicio* a trozos (Read · Token · Clean · Count), y por el camino entender tres cosas: que nada se modifica, que nada se ejecuta hasta la acción y que, gracias a eso, Spark reescribe tu plan antes de ejecutarlo.

## Bloque 1 · Inmutabilidad

Ninguna operación modifica el DataFrame original. Reasignar la variable no es lo mismo que cambiar el objeto: como en Python, `x = 5; x = x + 1` no cambia el 5, crea un 6.

In [ ]:
%python
df = spark.range(0, 20, 1, numPartitions=4).selectExpr(
    "id AS col1", "id * 2 AS col2", "id * 3 AS col3")

df_original = df
df = df.select("col1", "col2")          # objeto NUEVO: el original no se toca

print("Columnas de df_original:", df_original.columns)   # sigue con las 3
print("Columnas de df:         ", df.columns)            # solo 2
print("¿Es el mismo objeto?    ", df is df_original)     # False

In [ ]:
%python
# Lo mismo con un filter: el original no pierde filas.
# count() es una acción (lanza trabajo): aquí solo la usamos para comprobarlo; la explicamos en el bloque 5.
base = spark.range(0, 10)
filtrado = base.filter("id >= 5")

print("Filas en base:    ", base.count())      # sigue teniendo 10
print("Filas en filtrado:", filtrado.count())  # 5

## Bloque 2 · Práctica 1 · Leer el texto (Read)

Una fila por línea del libro, en una sola columna: `value`. `printSchema()` responde al instante: Spark ya sabe la forma del resultado sin haber procesado el texto. El análisis no espera; la ejecución, sí.

In [ ]:
%python
texto = [
    "The Project Gutenberg EBook of Pride and Prejudice, by Jane Austen",
    "",
    "This eBook is for the use of anyone anywhere at no cost and with",
    "almost no restrictions whatsoever.  You may copy it, give it away or",
    "re-use it under the terms of the Project Gutenberg License included",
    "with this eBook or online at www.gutenberg.org",
    "",
    "Title: Pride and Prejudice",
    "",
    "It is a truth universally acknowledged, that a single man in",
    "possession of a good fortune, must be in want of a wife.",
    "",
    "However little known the feelings or views of such a man may be",
    "on his first entering a neighbourhood, this truth is so well",
    "fixed in the minds of the surrounding families, that he is",
    "considered the rightful property of some one or other of their",
    "daughters.",
]

# Hoy el texto está en el notebook, es decir, en el driver: es pequeño.
# Con un fichero real, spark.read.text(ruta) lo leen los executors directamente del almacenamiento.
book = spark.createDataFrame([(línea,) for línea in texto], ["value"])
book.printSchema()
# root
#  |-- value: string (nullable = true)

## Bloque 3 · Práctica 2 · Transformaciones: nada se ejecuta (Token · Clean)

Cinco transformaciones narrow. Cada una devuelve un DataFrame nuevo y solo añade un paso al plan: ninguna celda lee el libro.

In [ ]:
%python
from pyspark.sql.functions import col, split, explode, lower, regexp_extract

lines = book.select(split(col("value"), " ").alias("line"))          # Token
words = lines.select(explode(col("line")).alias("word"))
words_lower = words.select(lower(col("word")).alias("word_lower"))   # Clean
words_clean = words_lower.select(
    regexp_extract(col("word_lower"), "[a-z]+", 0).alias("word")
)
words_nonull = words_clean.filter(col("word") != "")

In [ ]:
%python
# El plan crece; el libro aún no se ha leído. explain() no es una acción.
words_nonull.explain(mode="formatted")

## Bloque 4 · Cómo reescribe Spark tu plan

Como nada se ejecuta hasta la acción, Spark tiene la cadena **entera** delante y puede optimizarla. `explain(mode="extended")` enseña los cuatro planes por los que pasa:

| Plan | Qué hace |
|---|---|
| **Parsed** | Traduce tu código tal cual, sin comprobar nada. |
| **Analyzed** | Comprueba que las columnas existen y los tipos cuadran. |
| **Optimized** | Catalyst reordena, combina y elimina pasos. Sigue siendo lógico. |
| **Physical** | Cómo se ejecuta de verdad: operadores, stages y tasks. |

Vamos a escribir a propósito un plan **malo**: filtros repetidos, una columna que no se usa y un filtro sobre una columna derivada.

In [ ]:
%python
from pyspark.sql import functions as F

df = (spark.range(1000)                              # id: 0..999
      .withColumn("v", F.rand(seed=42))              # aleatorio entre 0 y 1
      .withColumn("w", F.col("v") * 2))              # derivada

# Cadena de solo transformaciones narrow, con filtros redundantes
df_plan = (df
    .filter(F.col("v") > 0.10)
    .filter(F.col("v") > 0.15)
    .filter(F.col("v") > 0.20)
    .filter(F.col("v") < 0.95)
    .filter((F.col("id") % 2) == 0)
    .withColumn("z", F.col("w") + 1)
    .select("id", "z")                               # proyección: w y v ya no se piden
    .filter(F.col("z") > 1.2))

In [ ]:
%python
# Los cuatro planes, de arriba abajo
df_plan.explain(mode="extended")

In [ ]:
%python
# La misma consulta, escrita limpia: un solo filtro y sin la columna w
limpio = (df
    .filter((F.col("v") > 0.20) & (F.col("v") < 0.95) & ((F.col("id") % 2) == 0))
    .select("id", (F.col("v") * 2 + 1).alias("z"))
    .filter(F.col("z") > 1.2))

limpio.explain()      # solo el plan físico: compáralo con el de df_plan

## Bloque 5 · Receta perezosa: leer → transformar → escribir

Otra receta corta, con un CSV pequeño en el volumen `lab0702.ejercicios.ficheros`. Montamos los pasos y **no pasa nada** hasta el `write`: esa es la acción que lanza el trabajo.

In [ ]:
%python
from uuid import uuid4

RUTA_VOLUMEN = "/Volumes/lab0702/ejercicios/ficheros"
try:
    dbutils.fs.ls(RUTA_VOLUMEN)
except Exception as e:
    raise FileNotFoundError(f"No existe {RUTA_VOLUMEN}: ejecuta antes 00_preparar_entorno.") from e

ROOT = f"{RUTA_VOLUMEN}/{uuid4().hex}"
entrada, salida = ROOT + "/entrada", ROOT + "/resultado"
print("Directorio de esta ejecución:", ROOT)

In [ ]:
%python
from pyspark.sql import functions as F

# Preparación: un CSV de ejemplo en el volumen (esto sí escribe: write es una acción)
spark.range(0, 120, 1, numPartitions=4).selectExpr(
    "id % 3 AS categoria", "id AS importe"
).write.csv(entrada, header=True)

# Receta: leer → transformar → agrupar. Todavía no se calcula nada.
df = spark.read.schema("categoria LONG, importe LONG").option("header", True).csv(entrada)
preparado = df.withColumn("doble", F.col("importe") * 2).where(F.col("doble") >= 120)
resultado = preparado.groupBy("categoria").count()
resultado.explain(mode="formatted")

In [ ]:
%python
# Acción: el write lanza el trabajo
resultado.write.csv(salida, header=True)

leido = spark.read.schema("categoria LONG, count LONG").option("header", True).csv(salida)
print({r.categoria: r["count"] for r in leido.collect()})
# Esperado: {0: 20, 1: 20, 2: 20}

display(dbutils.fs.ls(salida))

## Bloque 6 · Práctica 3 · `count()` lanza el job (Count)

Volvemos al wordcount. Hasta ahora Spark solo tomaba notas. `count()` pide un número y **nace un job**.

In [ ]:
%python
words_nonull.count()                              # acción: nace un job

In [ ]:
%python
counts = words_nonull.groupBy("word").count()     # transformación wide: aún nada
print(counts)  # DataFrame[word: string, count: bigint]

`groupBy("word").count()` **no** es una acción: devuelve un DataFrame. El `count` de dentro es un paso más del plan.

In [ ]:
%python
# Otro job, el top 10 (con shuffle: el groupBy es wide)
counts.orderBy(col("count").desc()).show(10)

### Dos `count` distintos

No es lo mismo contar **filas** que contar **grupos**. Y `count(columna)` ignora los nulos, mientras que `count(*)` cuenta filas.

In [ ]:
%python
from pyspark.sql import functions as F

df = spark.range(0, 120, 1, numPartitions=4).selectExpr(
    "id % 3 AS categoria",
    "CASE WHEN id % 2 = 0 THEN id END AS valor",
)

resultado = df.groupBy("categoria").count()
print("Columnas (aún no se ha ejecutado):", resultado.columns)

n_grupos = resultado.count()   # acción: cuenta grupos, no filas
n_filas = df.count()           # acción: cuenta filas

print("Grupos:", n_grupos)     # 3
print("Filas: ", n_filas)      # 120
print("Filas por grupo:", {r.categoria: r["count"] for r in resultado.collect()})
print("Valores no nulos:", df.select(F.count("valor")).first()[0])  # 60

## Extra · 8 particiones

En serverless no eliges los núcleos ni hay Spark UI, pero sí puedes crear **8 particiones** y ver cómo se reparte el dato con `spark_partition_id`. Con 4 núcleos, 8 particiones irían en dos oleadas.

In [ ]:
%python
from pyspark.sql.functions import spark_partition_id, sum as spark_sum

df8 = spark.range(0, 8000, 1, numPartitions=8)
df8.explain(mode="formatted")

reparto = (df8
           .select(spark_partition_id().alias("particion"))
           .groupBy("particion")
           .count()
           .orderBy("particion"))
reparto.show()

print("Suma de id:", df8.agg(spark_sum("id")).first()[0])

## Recap

| Bloque | Qué te llevas |
|---|---|
| Inmutabilidad | Reasignar no modifica: toda operación devuelve un DataFrame nuevo |
| Modelo perezoso | Las transformaciones anotan; `count()`, `show()` o `write` ejecutan |
| El plan | Con la cadena entera delante, Catalyst funde filtros y quita columnas que no usas |
| Una acción, un job | Y dentro, stages y tasks: lo abrimos el 23.11 con el DAG y el Query Profile |